# 01: Dataset Overview & Structural Profiling
**Project:** F1 Telemetry Intelligence  
**Grand Prix:** 2024 Abu Dhabi Grand Prix (Yas Marina Circuit)  
**Phase:** Phase 5 – Exploratory Data Analysis (EDA)  

---

## 1. Objective
Inspect processed dataset dimensions, schemas, driver rosters, lap classifications, and temporal sampling frequencies.

## 2. Key Analytical Questions
- **Q1:** What are the dimensions, memory footprints, and schemas of the processed datasets?
- **Q2:** How are the 96 driver-lap sequences distributed across tactical classifications?
- **Q3:** What is the empirical telemetry sampling frequency and nominal Hz?

## 3. Environment & Architectural Rule Compliance
- Execution Environment: Local `.venv` Python kernel in VS Code.
- Data Protection: Processed datasets are loaded strictly in read-only mode; raw data in `data/raw/` is untouched.
- Reusable Engine: Imports analytical logic directly from `src/analysis/`.


## 4. Setup, Configuration & Reusable Imports

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Set headless matplotlib backend and temporary config
os.environ.setdefault("MPLCONFIGDIR", tempfile.gettempdir())
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "eda" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root configured: {PROJECT_ROOT}")

from src.analysis.data_loader import load_all_datasets
from src.analysis.dataset_overview import (
    compute_dataset_dimensions,
    compute_lap_classification_summary,
    compute_sampling_frequency_stats,
    plot_lap_classification_distribution,
    plot_sampling_interval_distribution
)


## 5. Data Ingestion & Dimensional Inventory
Loading processed telemetry, race laps, and resampled spatial grids.

In [ ]:
tel_df, laps_df, res_df = load_all_datasets()
dim_df = compute_dataset_dimensions(tel_df, laps_df, res_df)
dim_df


## 6. Tactical Lap Classification Analysis
Breakdown of 96 driver-lap sequences across green flag, VSC, standing start, and crash retirement.

In [ ]:
class_summary = compute_lap_classification_summary(tel_df)
class_summary


### Visualizing Lap Taxonomy Distribution

In [ ]:
fig_path = plot_lap_classification_distribution(tel_df)
from IPython.display import Image, display
display(Image(filename=str(fig_path)))


## 7. Empirical Telemetry Sampling Frequency
Calculating delta time (dt) between successive samples and empirical Hz.

In [ ]:
sampling_stats = compute_sampling_frequency_stats(tel_df)
print("Empirical Sampling Statistics:")
for k, v in sampling_stats.items():
    print(f"  {k}: {v}")

fig_samp_path = plot_sampling_interval_distribution(tel_df)
display(Image(filename=str(fig_samp_path)))


## 8. Findings, Limitations & Conclusion

### Key Findings
1. **Multi-Scale Architecture:**
   - Telemetry: 72,476 time-domain rows (37 columns) across 20 drivers.
   - Race Laps: 1,035 lap-level rows (48 columns) covering the entire 58-lap Grand Prix.
   - Resampled Grid: 495,849 synchronized spatial slices (18 columns) at 1.0m intervals.
2. **Operational Regimes:**
   - 38 Clean Flying Laps (Laps 4 & 5) provide representative green-flag benchmark pace.
   - 38 VSC-affected laps (Laps 2 & 3) capture delta-limited tactical pacing.
   - 19 Start Laps capture standing start acceleration and tyre warm-up dynamics.
   - 1 DNF sequence (Perez Lap 1) captures collision termination.
3. **Sampling Rigor:** Telemetry arrives at a steady median delta t of ~0.101 seconds (~9.9 Hz).

### Limitations
Telemetry ingestion currently spans the first 5 laps per driver. Laps 6 to 58 remain available in race-laps summary form.

### Conclusion
Dataset dimensions and classifications are validated and ready for kinematic analysis.
